In [ ]:
# ==========================================
# GOLD FACTOR RESEARCH
# NOTEBOOK 02 - FACTOR RESEARCH
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "../data/processed/eda_output.csv",
    index_col=0,
    parse_dates=True
)

print("Dataset Shape:", df.shape)

df.head()

# ==========================================
# VERIFY REQUIRED COLUMNS
# ==========================================

required_columns = [
    "gold",
    "spy",
    "silver",
    "future_return_5m"
]

for col in required_columns:
    print(f"{col}: {col in df.columns}")

# ==========================================
# MOMENTUM FACTORS
# ==========================================

df["mom_1m"] = (
    df["gold"]
    .pct_change(21)
)

df["mom_3m"] = (
    df["gold"]
    .pct_change(63)
)

df["mom_6m"] = (
    df["gold"]
    .pct_change(126)
)

print("\nMomentum Factors Created")

# ==========================================
# TREND FACTORS
# ==========================================

df["ma50"] = (
    df["gold"]
    .rolling(50)
    .mean()
)

df["ma200"] = (
    df["gold"]
    .rolling(200)
    .mean()
)

df["above_200ma"] = (
    df["gold"]
    >
    df["ma200"]
).astype(int)

print("Trend Factors Created")

# ==========================================
# VOLATILITY FACTOR
# ==========================================

df["gold_return"] = (
    df["gold"]
    .pct_change()
)

df["vol20"] = (
    df["gold_return"]
    .rolling(20)
    .std()
    *
    np.sqrt(252)
)

print("Volatility Factors Created")

# ==========================================
# RELATIVE STRENGTH FACTORS
# ==========================================

df["gold_vs_spy"] = (
    df["gold"]
    /
    df["spy"]
)

df["gold_vs_silver"] = (
    df["gold"]
    /
    df["silver"]
)

print("Relative Strength Factors Created")

# ==========================================
# FACTOR SUMMARY
# ==========================================

factor_columns = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "above_200ma",
    "vol20",
    "gold_vs_spy",
    "gold_vs_silver"
]

print("\nFactor Summary Statistics")

display(
    df[
        factor_columns
    ].describe().T
)

# ==========================================
# CORRELATION ANALYSIS
# ==========================================

corr_matrix = df[
    factor_columns +
    ["future_return_5m"]
].corr()

print("\nCorrelation With Future Returns")

display(
    corr_matrix[
        ["future_return_5m"]
    ].sort_values(
        by="future_return_5m",
        ascending=False
    )
)

# ==========================================
# CORRELATION HEATMAP
# ==========================================

plt.figure(
    figsize=(12,8)
)

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title(
    "Factor Correlation Matrix"
)

plt.tight_layout()

plt.savefig(
    "../images/factor_correlation_heatmap.png"
)

plt.show()

# ==========================================
# MOMENTUM FACTOR RESEARCH
# ==========================================

df["momentum_bucket"] = pd.qcut(
    df["mom_6m"],
    5,
    labels=False,
    duplicates="drop"
)

momentum_results = (
    df.groupby(
        "momentum_bucket"
    )["future_return_5m"]
    .mean()
)

print("\nMomentum Results")

display(momentum_results)

momentum_results.plot(
    kind="bar",
    figsize=(10,5)
)

plt.title(
    "Future Returns By Momentum Quintile"
)

plt.xlabel(
    "Momentum Quintile"
)

plt.ylabel(
    "Average Future 5-Month Return"
)

plt.tight_layout()

plt.savefig(
    "../images/momentum_factor.png"
)

plt.show()

# ==========================================
# TREND FACTOR RESEARCH
# ==========================================

trend_results = (
    df.groupby(
        "above_200ma"
    )["future_return_5m"]
    .mean()
)

print("\nTrend Results")

display(trend_results)

trend_results.plot(
    kind="bar",
    figsize=(8,5)
)

plt.title(
    "Future Returns Above vs Below 200 Day MA"
)

plt.ylabel(
    "Average Future Return"
)

plt.tight_layout()

plt.savefig(
    "../images/trend_factor.png"
)

plt.show()

# ==========================================
# VOLATILITY FACTOR RESEARCH
# ==========================================

df["volatility_bucket"] = pd.qcut(
    df["vol20"],
    5,
    labels=False,
    duplicates="drop"
)

volatility_results = (
    df.groupby(
        "volatility_bucket"
    )["future_return_5m"]
    .mean()
)

print("\nVolatility Results")

display(volatility_results)

volatility_results.plot(
    kind="bar",
    figsize=(10,5)
)

plt.title(
    "Future Returns By Volatility Regime"
)

plt.ylabel(
    "Average Future Return"
)

plt.tight_layout()

plt.savefig(
    "../images/volatility_factor.png"
)

plt.show()

# ==========================================
# RELATIVE STRENGTH RESEARCH
# ==========================================

df["relative_strength_bucket"] = pd.qcut(
    df["gold_vs_spy"],
    5,
    labels=False,
    duplicates="drop"
)

rs_results = (
    df.groupby(
        "relative_strength_bucket"
    )["future_return_5m"]
    .mean()
)

print("\nRelative Strength Results")

display(rs_results)

rs_results.plot(
    kind="bar",
    figsize=(10,5)
)

plt.title(
    "Future Returns By Gold/SPY Relative Strength"
)

plt.ylabel(
    "Average Future Return"
)

plt.tight_layout()

plt.savefig(
    "../images/relative_strength_factor.png"
)

plt.show()

# ==========================================
# FACTOR RANKINGS
# ==========================================

factor_rankings = pd.DataFrame(
{
    "Factor": factor_columns,

    "Correlation": [
        df[f].corr(
            df["future_return_5m"]
        )
        for f in factor_columns
    ]
}
)

factor_rankings = (
    factor_rankings
    .sort_values(
        by="Correlation",
        ascending=False
    )
)

print("\nFactor Rankings")

display(factor_rankings)

# ==========================================
# FACTOR IMPORTANCE CHART
# ==========================================

plt.figure(
    figsize=(10,6)
)

plt.barh(
    factor_rankings["Factor"],
    factor_rankings["Correlation"]
)

plt.title(
    "Factor Predictive Strength"
)

plt.xlabel(
    "Correlation With Future Return"
)

plt.tight_layout()

plt.savefig(
    "../images/factor_importance.png"
)

plt.show()

# ==========================================
# EXPORT DATASET
# ==========================================

df.to_csv(
    "../data/processed/factor_data.csv"
)

print(
    "\nfactor_data.csv exported successfully."
)

# ==========================================
# RESEARCH SUMMARY
# ==========================================

print("\n" + "=" * 60)
print("FACTOR RESEARCH SUMMARY")
print("=" * 60)

print("""
Objective:
Identify quantitative factors that help predict
whether Gold will appreciate by more than 5%
over the next five months.

Factors Tested:
- Momentum
- Trend
- Volatility
- Relative Strength

Deliverables Produced:
- Correlation Matrix
- Momentum Analysis
- Trend Analysis
- Volatility Analysis
- Relative Strength Analysis
- Factor Rankings
- Research Charts
- factor_data.csv

Next Notebook:
03_information_diffusion.ipynb
""")